In [41]:
import torch
import gymnasium as gym
import random
from collections import deque
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import queue


device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [42]:
class Qnet(nn.Module):
    def __init__(self, grid_size=10, channels = 6, actions=4):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, 32, kernel_size = 3, padding=1), nn.ReLU(),
            nn.Conv2d(32,64,kernel_size = 3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2,2),
            nn.Conv2d(64,64,kernel_size = 3,padding=1), nn.ReLU(),
            nn.Flatten(),
            nn.Linear(64 * grid_size//2 * grid_size//2, 256), nn.ReLU(),
            nn.Linear(256,actions),
        )
    def forward(self,s):
        
        return self.net(s)
class Buffer:
    def __init__(self, cap = 100000):
        self.buffer = deque(maxlen=cap)
    def add(self, state,action,reward,nextstate,done):
        self.buffer.append((state,action,reward,nextstate,done))
    def sample(self,bsize):
        batch = random.sample(self.buffer,bsize)
        states,actions,rewards,nextstates,dones = zip(*batch)
        return (
            torch.tensor(np.array(states),dtype=torch.float32),
            torch.tensor(actions,dtype=torch.long),
            torch.tensor(rewards,dtype=torch.float32),
            torch.tensor(np.array(nextstates),dtype=torch.float32),
            torch.tensor(dones,dtype=torch.float32),
        )
    def __len__(self):
        return len(self.buffer)
    
        

In [63]:

class SnakeEnv(gym.Env):
    DIRS = [(0, -1), (1, 0), (0, 1), (-1, 0)]

    def __init__(self, grid_size=10, snake_count = 8, apple_count = 20, max_steps_without_apple=100):
        super().__init__()
        self.grid_size = grid_size
        self.max_steps_without_apple = max_steps_without_apple
        self.snake_count = snake_count
        self.apple_count = apple_count
        
        self.action_space = gym.spaces.Discrete(4)
            
        self.observation_space = gym.spaces.Box(
            low=-1.0, high=20.0, shape=(snake_count, 5,grid_size,grid_size), dtype=np.float32
        )

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.snakes = []
        self.apples = []
        for _ in range(self.snake_count):
            self.snakes.append([self._spawn_point()])
        for _ in range(self.apple_count):
            self.apples.append(self._spawn_point())
        self.directions = [1] * self.snake_count
        self.steps_since_apple = [0] * self.snake_count
        self.steps = [0] * self.snake_count
        self.alive = [True] * self.snake_count
        return self._obs(), {}

    def _respawn_snake(self, snake_id):
        self.alive[snake_id] = True
        self.snakes[snake_id] = [self._spawn_point()]
        self.directions[snake_id] = 1
        self.steps_since_apple[snake_id] = 0

   
    def step(self, actions):

        snakes1 = self.snakes
        r = [0] * self.snake_count
        term = [False] * self.snake_count
        trun = [False] * self.snake_count
        for i in range(self.snake_count):
            
            if len(self.snakes[i]) > 1 and (actions[i] + 2) % 4 == self.directions[i]:
                actions[i] = self.directions[i]
        
            self.directions[i] = actions[i]
            dx, dy = self.DIRS[actions[i]]
            head_x, head_y = snakes1[i][0]
            new_head = (head_x + dx, head_y + dy)
            
            if not (0 <= new_head[0] < self.grid_size and 0 <= new_head[1] < self.grid_size):
                term[i] = True
                self.alive[i] = False
                r[i] = -1
                continue

            snakes1[i] = [new_head] + self.snakes[i]


        for i in range(self.snake_count):
            if self.alive[i]:
                head_x, head_y = snakes1[i][0]
                for j in range(self.snake_count):
                    if j != i and snakes1[i][0] in snakes1[j]:
                        term[i] = True
                        self.alive[i] = False
                        r[i] = -1
                        break
                if self.alive[i] == False: continue
                new_head = snakes1[i][0]
                if new_head in self.apples:
                    r[i] = 1
                    self.steps_since_apple[i] = 0
                  
                else:
                    apple = ()
                    dist = 100000
                    for (x,y) in self.apples:
                        dist1 = abs(new_head[0] - x) + abs(new_head[1] - y)
                        if dist > dist1:
                            dist = dist1
                            apple = (x,y)
                    old_dist = abs(apple[0] - snakes1[i][1][0]) + abs(apple[1] - snakes1[i][1][1])
                    new_dist = abs(apple[0] - new_head[0]) + abs(apple[1] - new_head[1])
                    snakes1[i].pop()
                    r[i] = -0.01 + 0.1 * (old_dist - new_dist)
                    self.steps_since_apple[i] += 1
            
                self.snakes[i] = snakes1[i]
                self.steps[i] += 1
               
            
               
                trun[i] = self.steps_since_apple[i] > self.max_steps_without_apple
        for i in range(self.snake_count):
            if self.alive[i] == False:
                for j in self.snakes[i]:
                    self.apples.append(j)
                self.snakes[i].clear()
                self._respawn_snake(i)
            elif self.steps_since_apple[i] == 0:
                apple = ()
                for j in self.apples:
                    if j == self.snakes[i][0]:
                        apple = j
                self.apples.remove(apple)
                if len(self.apples) < self.apple_count:
                    self.apples.append(self._spawn_point())
        return self._obs(), r, term, trun, {}

    def _spawn_point(self):
        for i in range(100):
            pos = (
                self.np_random.integers(0, self.grid_size),
                self.np_random.integers(0, self.grid_size),
            )
            a = any(pos in snake for snake in self.snakes)
            b = pos in self.apples
            if not a and not b:
                return pos
        return (0,0)
   
    def _obs(self):
  
        obs_list = []
        for i in range(self.snake_count):
            head_x, head_y = self.snakes[i][0]
    
            lol = np.zeros((6,self.grid_size,self.grid_size),dtype=np.float32)
    
    
            lol[0,head_x,head_y] = 1
    
            for (x,y) in self.snakes[i][1:]: 
                lol[1,x,y] = 1

            for (x,y) in self.apples:
                lol[2,x,y] = 1
           
            lol[3,0,:]=1
            lol[3,:,0]=1
            lol[3,self.grid_size - 1,:] = 1
            lol[3,:,self.grid_size - 1] = 1

            for j in range(self.snake_count):
                if i != j and self.alive[j] == True:
                    lol[5,self.snakes[j][0][0],self.snakes[j][0][1]] = 1
                    for lp in self.snakes[j][1:]:
                        lol[4,lp[0],lp[1]] = 1
            obs_list.append(lol)
        return np.stack(obs_list,axis=0)



In [64]:
snake_count = 4
env = SnakeEnv(grid_size=50, snake_count=snake_count)

In [65]:
Q = Qnet(grid_size = 50, channels = 6,actions=4).to(device)
Qtarget = Qnet(grid_size = 50, channels = 6 ,actions=4).to(device)
Qtarget.load_state_dict(Q.state_dict())
buffer = Buffer(cap=100000)

In [66]:
opt = torch.optim.Adam(Q.parameters(),lr=5e-4)
g = 0.99
eps = 1.0
ints = 1000
b = 32
pl = 1000
n = 100000
obs,_ = env.reset()

print(type(obs))
tau = 0.005
episode_return = [0 for _ in range(snake_count)]
episode_count = [0 for _ in range(snake_count)]
returns = [[] for _ in range(snake_count)]
best_avg = 0

for step in range(n):
    if random.random() < eps:
        actions = [env.action_space.sample() for _ in range(snake_count)]
    else:
        with torch.no_grad():
            
            obs_t = [torch.from_numpy(obs1).unsqueeze(0).to(device) for obs1 in obs]
            actions = [Q(a).argmax().item() for a in obs_t]
    nextobs, r, term, trun, _ = env.step(actions)
    done = [0 for _ in range(snake_count)]
    for i in range(snake_count):
        episode_return[i] += r[i]
        done[i] = term[i] or trun[i]
    
    for i in range(snake_count):
        buffer.add(obs[i],actions[i],r[i],nextobs[i],done[i])
    obs = [nextobs[i] for i in range(snake_count)]

    for i in range(snake_count):
        if done[i]:
            returns[i].append(episode_return[i])
            episode_count[i] += 1
            if episode_count[i] % 100 == 0:
                avg = sum(returns[i][-100:]) / 10
                if avg > best_avg:
                    best_avg = avg
                    torch.save(Q.state_dict(),"best.pt")
                print(f"{step} for SNAKE --- {i} ---, avg return (last 100): {avg:.1f}, epsilon: {eps:.3f}")
            episode_return[i] = 0
    if len(buffer) > ints:
        states, actions,rs,nexts,dones = buffer.sample(b)
        states, actions, rs, nexts, dones = [t.to(device) for t in ( states, actions, rs, nexts, dones)]
        q_values = Q(states)
        predicted = q_values.gather(1, actions.unsqueeze(1)).squeeze(1)
        
        with torch.no_grad():
            next_actions = Q(nexts).argmax(dim=1,keepdim=True).to(device)
            next_q = Qtarget(nexts).gather(1,next_actions).squeeze(1).to(device)
            target = rs + g * next_q * (1 - dones)
        loss = F.smooth_l1_loss(predicted,target)
        #if step % 100 == 0 : print(step,loss.item())
        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(Q.parameters(), 10.0)   
        opt.step()
    with torch.no_grad():
        for tp, op in zip(Qtarget.parameters(), Q.parameters()):
            tp.data.mul_(1 - tau).add_(op.data, alpha=tau)
    eps = max(0.05, eps * 0.9995)

<class 'numpy.ndarray'>
269 for SNAKE --- 0 ---, avg return (last 100): -0.4, epsilon: 0.874
441 for SNAKE --- 2 ---, avg return (last 100): 0.4, epsilon: 0.802
746 for SNAKE --- 2 ---, avg return (last 100): 0.2, epsilon: 0.689
779 for SNAKE --- 3 ---, avg return (last 100): 0.6, epsilon: 0.677
845 for SNAKE --- 1 ---, avg return (last 100): 1.3, epsilon: 0.655
879 for SNAKE --- 3 ---, avg return (last 100): 0.1, epsilon: 0.644
1140 for SNAKE --- 0 ---, avg return (last 100): 0.2, epsilon: 0.565
1240 for SNAKE --- 0 ---, avg return (last 100): 0.1, epsilon: 0.538
1251 for SNAKE --- 2 ---, avg return (last 100): 0.4, epsilon: 0.535
1693 for SNAKE --- 3 ---, avg return (last 100): 1.6, epsilon: 0.429
3095 for SNAKE --- 3 ---, avg return (last 100): 11.2, epsilon: 0.213
3195 for SNAKE --- 3 ---, avg return (last 100): -0.1, epsilon: 0.202
3868 for SNAKE --- 3 ---, avg return (last 100): 5.7, epsilon: 0.144
4023 for SNAKE --- 1 ---, avg return (last 100): 24.0, epsilon: 0.134
4268 for SNA

KeyboardInterrupt: 